In [ ]:
import pandas as pd
fp = "../data/sba_loans_prepared/sba_loans_stage1.csv"
df = pd.read_csv(fp)

In [ ]:
df

In [ ]:
target_recode = {"PIF": 0, "CHGOFF": 1}
df["LoanStatus"] = df["LoanStatus"].replace(target_recode)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

def generate_tvt_indices(data_size, test_ratio=0.2, val_ratio=0.2, random_state=None):
    """
    Generates train, validation, and test indices for a dataset.

    Args:
        data_size (int): The total number of samples in the dataset.
        test_ratio (float): The proportion of the dataset to allocate to the test set.
        val_ratio (float): The proportion of the *remaining* data (after test split)
                           to allocate to the validation set.
        random_state (int, optional): Seed for the random number generator for reproducibility.

    Returns:
        tuple: A tuple containing three NumPy arrays:
               (train_indices, val_indices, test_indices)
    """

    # Create a range of indices representing the entire dataset
    indices = np.arange(data_size)

    # First split: separate out the test set
    train_val_indices, test_indices = train_test_split(
        indices, test_size=test_ratio, random_state=random_state
    )

    # Second split: separate out the validation set from the remaining train_val_indices
    # The validation ratio here is applied to the *remaining* data after the test split.
    # To get the correct proportion relative to the original dataset, we adjust it.
    # For example, if test_ratio=0.2 and val_ratio=0.2, then the validation set will be
    # 0.2 * (1 - 0.2) = 0.16 of the original data.
    adjusted_val_ratio = val_ratio / (1 - test_ratio)

    train_indices, val_indices = train_test_split(
        train_val_indices, test_size=adjusted_val_ratio, random_state=random_state
    )

    return train_indices, val_indices, test_indices

# Example Usage:
# data_size = 1000
# train_idx, val_idx, test_idx = generate_tvt_indices(data_size, test_ratio=0.2, val_ratio=0.2, random_state=42)
# print(f"Train indices length: {len(train_idx)}")
# print(f"Validation indices length: {len(val_idx)}")
# print(f"Test indices length: {len(test_idx)}")

In [ ]:
train_ind, val_ind, test_ind = generate_tvt_indices(df.shape[0], .15, .15, random_state=42)

In [ ]:
cols = df.columns.to_list()

In [ ]:
NUM_COLS = ['LoanStatus', 'GrossChargeOffAmount', 'NumPmtsMade',  'GrossApproval', 'InitialInterestRate']
data_types = {k: 'category' for k in cols if k not in NUM_COLS}

In [ ]:
ID_COLS = ["BorrName", "BankFDICNumber", "LoanID"]
df_id = df[ID_COLS]

In [ ]:
df = df.astype(data_types)

In [ ]:
EXCLUDE_COLS = NUM_COLS + ID_COLS
cat_cols = [c for c in cols if c not in EXCLUDE_COLS]
cat_cols = [c for c in cols if c not in EXCLUDE_COLS]

In [ ]:
cat_cols

In [ ]:
from category_encoders import *
loan_status = df["LoanStatus"]

In [ ]:
NUM_PREDS = [c for c in NUM_COLS if c not in["LoanStatus", "GrossChargeOffAmount"]]

In [ ]:
NUM_PREDS

In [ ]:
df_cat_train = df[df.index.isin(train_ind)][cat_cols]
y_train = loan_status[loan_status.index.isin(train_ind)]
df_cat_test = df[df.index.isin(test_ind)][cat_cols]
y_test = loan_status[loan_status.index.isin(test_ind)]
df_cat_val = df[df.index.isin(val_ind)][cat_cols]
y_val = loan_status[loan_status.index.isin(val_ind)]

In [ ]:
df_num_train = df[df.index.isin(train_ind)][NUM_PREDS]
df_num_test = df[df.index.isin(test_ind)][NUM_PREDS]
df_num_val = df[df.index.isin(val_ind)][NUM_PREDS]


**Note** The encoding of numerical attributes uses **only** the training dataset

In [ ]:
enc = TargetEncoder(cols=cat_cols, min_samples_leaf=20, smoothing=10).fit(df_cat_train, y_train)
enc_dataset_train = enc.transform(df_cat_train)
enc_dataset_test = enc.transform(df_cat_test)
enc_dataset_val = enc.transform(df_cat_val)

In [ ]:
df_train = pd.concat([enc_dataset_train,df_num_train], axis = 1)
df_test = pd.concat([enc_dataset_test,df_num_test], axis = 1)
df_val =  pd.concat([enc_dataset_val,df_num_val], axis = 1)

In [ ]:
df_train.loc[:, "LoanStatus"] = y_train.values
df_val.loc[:, "LoanStatus"] = y_val.values
df_test.loc[:, "LoanStatus"] = y_test.values

In [ ]:
df_val

In [ ]:
train_info_df = df[df.index.isin(train_ind)]
testinfo_df = df[df.index.isin(test_ind)]
val_info_df = df[df.index.isin(val_ind)]

**Note** The scaling of attributes uses **only** the training dataset

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
preds = [ c for c in df_train.columns.tolist() if c != "LoanStatus"]
X_train = df_train[preds]
scaler.fit(X_train)
X_train_std = scaler.transform(X_train)
df_train = pd.DataFrame(X_train_std)
df_train.columns = preds
df_train.loc[:, "LoanStatus"] = y_train.values

In [ ]:
df_train.shape

In [ ]:
X_test = df_test[preds]
X_test_std = scaler.transform(X_test)
df_test = pd.DataFrame(X_test_std, columns = preds)
df_test.loc[:, "LoanStatus"] = y_test.values

In [ ]:
X_val = df_val[preds]
X_val_std = scaler.transform(X_val)
df_val = pd.DataFrame(X_val_std, columns = preds)
df_val.loc[:, "LoanStatus"] = y_val.values

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_train.csv"
df_train.to_csv(fp, index=False)

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test.to_csv(fp, index=False)

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
df_val.to_csv(fp, index=False)

In [ ]:
fp_train = "../data/sba_loans_prepared/sba_train_raw.csv"
train_info_df = train_info_df.reset_index(drop=True)
train_info_df.to_csv(fp_train, index=False)
fp_val = "../data/sba_loans_prepared/sba_val_borr_raw.csv"
val_info_df = val_info_df.reset_index(drop=True)
val_info_df.to_csv(fp_val, index=False)
fp_test = "../data/sba_loans_prepared/sba_test_raw.csv"
testinfo_df = testinfo_df.reset_index(drop=True)
testinfo_df.to_csv(fp_val, index=False)

In [ ]:
df_train.LoanStatus.value_counts()


In [ ]:
train_info_df.LoanStatus.value_counts()

In [ ]:
df_val